# Fibromyalgia RAG Pipeline (v2 - layout-aware parsing)

A Retrieval-Augmented Generation (RAG) pipeline built over a single biomedical
review article: *"Fibromyalgia: A Review of the Pathophysiological Mechanisms
and Multidisciplinary Treatment Strategies"* (Jurado-Priego et al., 2024,
*Biomedicines* 12, 1543).

**What changed from v1, and why**

The previous version parsed sections with `text.find("2. Epidemiology")` on a
whitespace-collapsed string, and dropped the reference list entirely. That
works only for this one article's exact heading text and throws away every
citation. This version instead:

1. Extracts **PDF blocks with font metadata** (font family, size) instead of
   plain text, so numbered headings are detected by *how they're
   typeset* (bold for `N.`, italic for `N.N.`) rather than by hardcoded
   strings. This generalizes to any MDPI-style review article with the same
   numbering convention, not just this one PDF.
2. **Parses the reference list into 148 structured, numbered entries**
   instead of discarding it, so an in-text marker like `[12,45]` can be
   resolved back to the actual source at retrieval time.
3. **Chunks by sentence, not by raw character count**, and protects
   `[12,45]`-style citation markers during splitting so a chunk never ends
   mid-citation.
4. Fixes hyphenation rejoining at the correct point in the pipeline (the v1
   regex ran too late to ever match anything, so words like
   "musculoskeletal" stayed broken as "muscu- loskeletal").

**Pipeline stages**

1. Layout-aware PDF extraction (PyMuPDF, block/font level).
2. Heading detection from numbering + font weight, building a section tree.
3. Reference-list parsing into structured, numbered entries.
4. Citation-aware chunking (sentence-safe, marker-safe).
5. Embedding + FAISS indexing.
6. Retrieval with citation resolution (chunk -> which sources it actually cites).
7. Retrieval evaluation (Precision@K keyword benchmark).
8. Grounded answer generation (RAG generation step).


## 1. Setup

In [ ]:
%pip install -q pymupdf langchain-core langchain-text-splitters \
    langchain-community langchain-huggingface faiss-cpu sentence-transformers openai


In [ ]:
import os
import re
from pathlib import Path

# Project-relative paths (works locally, in Colab, and in CI alike)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_RAW_DIR / "biomedicines-12-01543.pdf"

assert PDF_PATH.exists(), (
    f"Source PDF not found at {PDF_PATH}. "
    "Place 'biomedicines-12-01543.pdf' in data/raw/ before running this notebook."
)
print("Using PDF:", PDF_PATH)


## 2. Layout-aware extraction

Instead of `page.get_text()` (a flat string with all layout information
discarded), we pull PyMuPDF's `"dict"` output, which gives every line its
**font name, size, and position**. This is what makes heading detection in
Section 3 possible without hardcoding heading strings.

In [ ]:
import fitz  # PyMuPDF

JUNK_LINE_PATTERNS = [
    r'^Biomedicines\s+\d{4},\s*\d+,?\s*(x FOR PEER REVIEW|\d+)$',  # repeated citation banner
    r'^\d+\s+of\s+\d+$',                                             # "4 of 22" page markers
]

def extract_lines(pdf_path: Path):
    """Extract text lines with font metadata (layout-aware, not plain text)."""
    doc = fitz.open(pdf_path)
    lines = []
    for pno, page in enumerate(doc):
        d = page.get_text("dict")
        for block in d["blocks"]:
            if block["type"] != 0:  # skip images
                continue
            for line in block["lines"]:
                spans = line["spans"]
                if not spans:
                    continue
                text = "".join(s["text"] for s in spans).strip()
                if not text:
                    continue
                lines.append({
                    "page": pno,
                    "text": text,
                    "fonts": {s["font"] for s in spans},
                    "y": line["bbox"][1],
                })
    doc.close()
    return [l for l in lines if not any(re.match(p, l["text"]) for p in JUNK_LINE_PATTERNS)]

lines = extract_lines(PDF_PATH)
print(f"Extracted {len(lines)} text lines (after removing running headers/footers)")
print(lines[5])


## 3. Heading Detection

Numbered headings in this article follow a consistent typographic
convention (verified against the actual PDF fonts, not assumed):

| Level | Pattern | Font |
|---|---|---|
| 1 | `N. Title` | Bold |
| 2 | `N.N. Title` | Italic |
| 3 | `N.N.N. Title` | Regular, but on its own line |

Detecting headings this way means the parser doesn't need to know the
article's actual section names in advance -- it will find `8. New Section`
in a different paper just as reliably as `2. Epidemiology` here.

In [ ]:
H1 = re.compile(r'^(\d{1,2})\.\s+(.+)$')
H2 = re.compile(r'^(\d{1,2}\.\d{1,2})\.\s+(.+)$')
H3 = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\.\s+(.+)$')

def _is_bold(block):
    return any('Bold' in f for f in block["fonts"])

def _is_italic(block):
    return any('Ital' in f for f in block["fonts"])

def detect_headings(lines):
    headings = []
    for i, b in enumerate(lines):
        t = b["text"]
        m3 = H3.match(t)
        m2 = H2.match(t) if not m3 else None
        m1 = H1.match(t) if not (m2 or m3) else None
        if m3:
            headings.append({"level": 3, "number": m3.group(1), "title": m3.group(2), "line_idx": i})
        elif m2 and _is_italic(b):
            headings.append({"level": 2, "number": m2.group(1), "title": m2.group(2), "line_idx": i})
        elif m1 and _is_bold(b):
            headings.append({"level": 1, "number": m1.group(1), "title": m1.group(2), "line_idx": i})
    return headings

headings = detect_headings(lines)
print(f"Detected {len(headings)} headings\n")
for h in headings:
    print(" " * ((h["level"] - 1) * 3), h["number"], h["title"])


## 4. Section Building

Slice the line stream between consecutive headings. Hyphenation rejoining
(`"muscu-" + "loskeletal"` -> `"musculoskeletal"`) happens **at join time**,
which is the only point where the original `-` + line-break pattern is still
visible -- doing it after lines are already space-joined (as v1 did) can
never match anything.

In [ ]:
def join_lines_dehyphenated(text_lines) -> str:
    out = ""
    for line in text_lines:
        if out.endswith("-") and line and line[0].islower():
            out = out[:-1] + line          # rejoin split word, no space
        elif out:
            out = out + " " + line
        else:
            out = line
    return out

def build_sections(lines, headings):
    sections = []
    for i, h in enumerate(headings):
        start = h["line_idx"] + 1
        end = headings[i + 1]["line_idx"] if i + 1 < len(headings) else len(lines)
        body_lines = [l["text"] for l in lines[start:end]]
        text = re.sub(r'\s+', ' ', join_lines_dehyphenated(body_lines)).strip()
        sections.append({
            "number": h["number"],
            "level": h["level"],
            "title": h["title"],
            "top_section": h["number"].split(".")[0],
            "text": text,
        })
    return sections

sections = build_sections(lines, headings)
for s in sections[:5]:
    print(f"[{s['number']:>6s}] {s['title']:<45s} -> {len(s['text']):5,d} chars")
print("...")
print(f"\nTotal sections/subsections: {len(sections)}")


## 5. Reference-List Parsing

The v1 pipeline discarded everything after "References". Here the reference
list is parsed into **148 individually addressable entries**, keyed by
citation number, so any `[N]` marker found in a chunk can be resolved back to
its source at retrieval time.

Note the entry-boundary regex requires a letter (not just an uppercase
letter) after the number, because some author surnames start lowercase
(e.g. entry 89, "da Rocha, A.P...."). This was caught by testing against the
real reference list, not assumed.

In [ ]:
REF_ENTRY = re.compile(r'\n(\d{1,3})\.\s+(?=[A-Za-z])')

def parse_references(pdf_path: Path) -> dict:
    doc = fitz.open(pdf_path)
    raw_text = "".join(page.get_text() + "\n" for page in doc)
    doc.close()

    m = re.search(r'\nReferences\n', raw_text)
    if not m:
        return {}
    ref_text = raw_text[m.end():]
    ref_text = ref_text.split("Disclaimer/Publisher")[0]
    ref_text = re.sub(r'Biomedicines\s+\d{4},\s*\d+,?\s*\d+\s*\n?\d*\s*of\s*\d+\s*\n?', '', ref_text)
    ref_text = re.sub(r'\n\d+\s+of\s+\d+\n', '\n', ref_text)

    matches = list(REF_ENTRY.finditer("\n" + ref_text))
    entries = {}
    for i, mm in enumerate(matches):
        num = int(mm.group(1))
        start = mm.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(ref_text) + 1
        content = ("\n" + ref_text)[start:end]
        entries[num] = re.sub(r'\s+', ' ', content).strip()
    return entries

references = parse_references(PDF_PATH)
print(f"Parsed {len(references)} reference entries (expected 148)")
missing = set(range(1, 149)) - set(references)
print("Missing entry numbers:", missing or "none")
print("\nExample [50]:", references[50][:140])
print("Example [89]:", references[89][:140], "  <- lowercase surname, verified not dropped")


## 6. Citation-Aware Chunking

Sentences are split without ever cutting inside a `[12,45]` marker (commas
inside brackets are temporarily masked before splitting, then restored).
Chunks are built up sentence-by-sentence to a soft character cap, so a
sentence -- and therefore a citation -- is never split across two chunks.
Each chunk also carries the list of reference numbers it actually cites.

In [ ]:
CITATION_MARKER = re.compile(r'\[([\d,\s]+)\]')

def split_sentences_keep_citations(text: str):
    protected = re.sub(r'\[([\d,\s\-]+)\]',
                        lambda mm: '[' + mm.group(1).replace(',', '\u00a7') + ']', text)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', protected)
    return [s.replace('\u00a7', ',') for s in sentences]

def extract_citation_numbers(text: str):
    nums = set()
    for m in CITATION_MARKER.finditer(text):
        for n in m.group(1).split(','):
            n = n.strip()
            if n.isdigit():
                nums.add(int(n))
    return sorted(nums)

def chunk_section(section: dict, max_chars: int = 900):
    sentences = split_sentences_keep_citations(section["text"])
    chunks, current = [], ""
    for sent in sentences:
        if current and len(current) + len(sent) + 1 > max_chars:
            chunks.append(current.strip())
            current = sent
        else:
            current = f"{current} {sent}".strip()
    if current:
        chunks.append(current.strip())
    return [
        {
            "text": c,
            "section_number": section["number"],
            "section_title": section["title"],
            "top_section": section["top_section"],
            "level": section["level"],
            "cited_refs": extract_citation_numbers(c),
        }
        for c in chunks
    ]

all_chunks = []
for s in sections:
    all_chunks.extend(chunk_section(s))

print(f"Total chunks: {len(all_chunks)}")
with_citations = [c for c in all_chunks if c["cited_refs"]]
print(f"Chunks carrying at least one citation: {len(with_citations)} / {len(all_chunks)}")
print()
sample = with_citations[0]
print(f"Sample chunk (section {sample['section_number']}, cites {sample['cited_refs']}):")
print(sample["text"][:300])
